# 2.5 SIMD 连续类矢量算子示例（Add 算子）
本示例基于 Ascend C SIMD 实现 Add 算子，帮助你快速上手。它完整呈现了 Device 端核函数实现、Host 端调用及编译运行的全流程，助你建立整体认知。

**Add 算子功能介绍**：数学表达式为 $z = x + y$，计算逻辑为逐元素完成 $z[i] = x[i] + y[i]$。


## 算子设计
**Device 端核函数**：
- 通过 `__global__` 修饰符声明核函数，`__vector__` 指定在矢量计算单元上执行。
- 数据分块（Tiling）：使用 `GetBlockIdx()` 确定每个 Block 负责处理的数据段。
- 数据搬入：通过 `DataCopy` 将输入从 Global Memory 搬至片上 UB。
- 数据计算：通过 `Add` 完成逐元素加法。
- 数据搬出：通过 `DataCopy` 将结果从 UB 搬回 Global Memory。

**Host 端运行时**：
- 使用 `aclrtMalloc` 分配 Device Memory，`aclrtMemcpy` 搬入输入数据。
- 通过 `<<<numBlocks, 0, stream>>>` 启动核函数，`aclrtSynchronizeStream` 同步等待。
- 使用 `aclrtMemcpy` 将计算结果从 Device Memory 拷回 Host Memory 验证。


## Device 端 Kernel 实现
后缀名为 `*.asc` 的代码文件包含 Host 端与 Device 端代码。首先包含所需头文件并定义常量：
- `kernel_operator.h`：提供 TPipe、TQue、LocalTensor、GlobalTensor、DataCopy、Add 等 C++ 基础 API。
- `acl/acl.h`：AscendCL 运行时接口（Host 侧）。
- `TOTAL_LENGTH = 8 * 2048`：总数据量，8 个核各处理 2048 个 float。


In [ ]:
!mkdir -p Sources/02.05

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment ready.")


In [ ]:
%%writefile Sources/02.05/c_api_add.asc

#include <cstdint>
#include <vector>
#include <iostream>
#include <algorithm>
#include "kernel_operator.h"
#include "acl/acl.h"

using namespace AscendC;

constexpr uint32_t TOTAL_LENGTH = 8 * 2048;
constexpr uint32_t NUM_BLOCKS = 8;



In [ ]:
%%writefile -a Sources/02.05/c_api_add.asc

__global__ __aicore__ void add_custom(__gm__ float* x, __gm__ float* y, __gm__ float* z)
{
    constexpr uint32_t blockLength = TOTAL_LENGTH / NUM_BLOCKS;
    uint32_t tileLength = blockLength;

    TPipe pipe;
    TQue<QuePosition::VECIN, 1> inQueueX, inQueueY;
    TQue<QuePosition::VECOUT, 1> outQueueZ;

    pipe.InitBuffer(inQueueX, 1, tileLength * sizeof(float));
    pipe.InitBuffer(inQueueY, 1, tileLength * sizeof(float));
    pipe.InitBuffer(outQueueZ, 1, tileLength * sizeof(float));

    uint32_t blockIdx = GetBlockIdx();
    GlobalTensor<float> xGm, yGm, zGm;
    xGm.SetGlobalBuffer(x);
    yGm.SetGlobalBuffer(y);
    zGm.SetGlobalBuffer(z);
    LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    DataCopy(xLocal, xGm[blockIdx * blockLength], tileLength);
    DataCopy(yLocal, yGm[blockIdx * blockLength], tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
    xLocal = inQueueX.DeQue<float>();
    yLocal = inQueueY.DeQue<float>();

    LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    Add(zLocal, xLocal, yLocal, tileLength);
    outQueueZ.EnQue(zLocal);
    zLocal = outQueueZ.DeQue<float>();
    DataCopy(zGm[blockIdx * blockLength], zLocal, tileLength);
    outQueueZ.FreeTensor(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}



核函数遵循**搬入 → 计算 → 搬回**三步法：

1. **初始化**：创建 `TPipe` 管理片上内存，通过 `InitBuffer` 为输入/输出队列分配 UB 空间。
2. **搬入**：用 `GetBlockIdx()` 获取当前核编号，计算数据偏移，`DataCopy` 将 GM 数据搬入 UB。
3. **计算**：`Add(zLocal, xLocal, yLocal, tileLength)` 在 UB 上执行逐元素加法。
4. **搬回**：`DataCopy` 将结果从 UB 搬回 GM。`EnQue/DeQue` 管理队列同步，`FreeTensor` 释放资源。


## Host 端代码实现
Host 端通过 `<<<>>>` 语法糖调用 Device 端核函数，完整流程为：
1. `aclInit` 初始化 → `aclrtSetDevice` 设置设备 → `aclrtCreateStream` 创建流。
2. `aclrtMalloc` 分配 Device Memory，`aclrtMemcpy` 将输入数据从 Host 拷入 Device。
3. `add_custom<<<8, nullptr, stream>>>(xd, yd, zd)` 启动 8 核并行计算。
4. `aclrtSynchronizeStream` 同步等待 → `aclrtMemcpy` 将结果拷回 Host 验证。


In [ ]:
%%writefile -a Sources/02.05/c_api_add.asc

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t total_length = TOTAL_LENGTH;
    std::vector<float> x(total_length), y(total_length);
    for (uint32_t i = 0; i < total_length; ++i) { x[i] = i * 0.1f; y[i] = i * 0.1f; }
    size_t bytes = total_length * sizeof(float);
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    float *xd=nullptr,*yd=nullptr,*zd=nullptr; uint8_t *zh=nullptr;
    aclrtMallocHost((void**)&zh, bytes);
    aclrtMalloc((void**)&xd, bytes, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&yd, bytes, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&zd, bytes, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xd, bytes, x.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yd, bytes, y.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE);
    add_custom<<<NUM_BLOCKS, nullptr, stream>>>(xd, yd, zd);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(zh, bytes, zd, bytes, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> out((float*)zh, (float*)(zh + bytes));
    bool ok = std::equal(out.begin(), out.end(), x.begin(), [&](float a, float b){ return a == b + b; });
    std::cout << (ok ? "[Success] verification passed." : "[Failed] verification failed!") << std::endl;
    aclrtFree(xd); aclrtFree(yd); aclrtFree(zd); aclrtFreeHost(zh);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0); aclFinalize();
    return 0;
}


## 编译与运行
用毕昇编译器编译，本课程面向 950 使用 `--npu-arch=dav-3510`：


In [ ]:
!bisheng Sources/02.05/c_api_add.asc --npu-arch=dav-3510 -o Sources/02.05/demo


In [ ]:
!if [ -e /dev/davinci0 ]; then Sources/02.05/demo; else cannsim record Sources/02.05/demo -s Ascend950 -o Sources/02.05/sim_out; fi


## 课后实践
仿照本例实现 ReLU 算子（提示：用 `Max` 与 0 取最大值）。


In [ ]:
!cat ./answer/02.05_answer.txt
